In [6]:
import os
import re
import json
import torch
import torchaudio
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from torchmetrics.text import WordErrorRate, CharErrorRate
from typing import Any, Dict, List, Union

In [7]:
MODEL_NAME = "openai/whisper-small"
JSON_LABELS_PATH = "/kaggle/input/datasets/vladyslavsydorak/toronto/dataset/labels.jsonl"
BATCH_SIZE = 8

test_lines = [
    'toronto_27', 'toronto_46', 'toronto_42', 'toronto_37', 'toronto_89',
    'toronto_43', 'toronto_157', 'toronto_9', 'toronto_156', 'toronto_7',
    'toronto_123', 'toronto_54', 'toronto_67', 'toronto_62', 'toronto_81',
    'toronto_134', 'toronto_148', 'toronto_21', 'toronto_135', 'toronto_166',
    'toronto_58'
]

In [8]:
class TorontoDataset(Dataset):
    def __init__(self, json_path: str, processor: WhisperProcessor, test_lines: List[str]):
        self.processor = processor
        self.sampling_rate = processor.feature_extractor.sampling_rate
        base_dir = "/kaggle/input/datasets/vladyslavsydorak/toronto"

        with open(json_path, 'r', encoding='utf-8') as f:
            data_dict = json.load(f)

        self.data = []
        missing_files_count = 0

        for relative_path, transcript in data_dict.items():
            full_path = os.path.join(base_dir, relative_path)

            if not os.path.exists(full_path):
                missing_files_count += 1
                continue

            is_test = any(test_id in full_path for test_id in test_lines)
            if is_test:
                self.data.append({"path": full_path, "text": transcript})

        if missing_files_count > 0:
            print(f"[TEST] Skipped {missing_files_count} missing files.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        waveform, sr = torchaudio.load(item["path"])
        
        if sr != self.sampling_rate:
            waveform = torchaudio.functional.resample(waveform, orig_freq=sr, new_freq=self.sampling_rate)

        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        input_features = self.processor.feature_extractor(
            waveform.squeeze().numpy(), 
            sampling_rate=self.sampling_rate, 
            return_tensors="pt"
        ).input_features[0]

        labels = self.processor.tokenizer(item["text"], return_tensors="pt").input_ids[0]
        return {"input_features": input_features, "labels": labels}

In [9]:
class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, processor: Any):
        self.processor = processor

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

In [10]:
class WhisperBaseEvalModule(pl.LightningModule):
    def __init__(self, model_name_or_path: str):
        super().__init__()
        
        self.processor = WhisperProcessor.from_pretrained(
            model_name_or_path, language="uk", task="transcribe"
        )
        self.model = WhisperForConditionalGeneration.from_pretrained(model_name_or_path)

        # Force the configuration for Ukrainian transcription
        self.model.generation_config.language = "uk"
        self.model.generation_config.task = "transcribe"
        self.model.generation_config.forced_decoder_ids = None

        self.wer_metric = WordErrorRate()
        self.cer_metric = CharErrorRate()

    def forward(self, input_features, labels=None):
        return self.model(input_features=input_features, labels=labels)

    def test_step(self, batch, batch_idx):
        input_features = batch["input_features"]
        labels = batch["labels"]

        attention_mask = torch.ones(
            input_features.shape[0],
            input_features.shape[2],
            dtype=torch.long,
            device=input_features.device
        )

        pred_ids = self.model.generate(
            input_features,
            attention_mask=attention_mask
        )

        labels[labels == -100] = self.processor.tokenizer.pad_token_id

        preds_str = self.processor.batch_decode(pred_ids, skip_special_tokens=True)
        labels_str = self.processor.batch_decode(labels, skip_special_tokens=True)

        preds_str = [self._normalize_text(text) for text in preds_str]
        labels_str = [self._normalize_text(text) for text in labels_str]

        self.wer_metric.update(preds_str, labels_str)
        self.cer_metric.update(preds_str, labels_str)

    def on_test_epoch_end(self):
        wer = self.wer_metric.compute()
        cer = self.cer_metric.compute()
        self.log("test_wer", wer)
        self.log("test_cer", cer)
        self.print(f"\nBase Model Test Results -> WER: {wer:.4f} | CER: {cer:.4f}")        
        self.wer_metric.reset()
        self.cer_metric.reset()

    def _normalize_text(self, text: str) -> str:
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

In [11]:
model = WhisperBaseEvalModule(model_name_or_path=MODEL_NAME)
processor = model.processor

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [12]:
test_dataset = TorontoDataset(JSON_LABELS_PATH, processor, test_lines=test_lines)
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=data_collator, 
    num_workers=2
)

[TEST] Skipped 10929 missing files.


In [13]:
trainer = pl.Trainer(
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    precision="16-mixed" if torch.cuda.is_available() else "32-true",
    logger=False
)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [14]:
trainer.test(model, dataloaders=test_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


Base Model Test Results -> WER: 0.3286 | CER: 0.1910